In [ ]:
import sympy as sp

sp.init_printing()

# ============================================================
# 1. Symbols
# ============================================================

c, mu0 = sp.symbols("c mu_0", positive=True, nonzero=True)
kappa = sp.symbols("kappa_EM", real=True)

phi = sp.symbols("phi", real=True)

a = sp.symbols("a", positive=True, nonzero=True)

A1, A2, A3 = sp.symbols("A_1 A_2 A_3", real=True)

Ex, Ey, Ez = sp.symbols("E_x E_y E_z", real=True)
Bx, By, Bz = sp.symbols("B_x B_y B_z", real=True)

# ============================================================
# 2. Basic definitions
# ============================================================

q = 1 - kappa*phi/c

eta = sp.diag(-1, 1, 1, 1)

# ============================================================
# 3. Tetrad e^a_mu
# ============================================================
#
# Coordinates:
# x^mu = (ct, x, y, z)
#
# Maxwell-adapted tetrad:
#
# e^0 = (1 - kappa_EM phi/c) dx^0
#       + kappa_EM A_i dx^i
#
# e^1 = a dx^1
# e^2 = a dx^2
# e^3 = a dx^3

e = sp.Matrix([
    [q,         kappa*A1, kappa*A2, kappa*A3],
    [0,         a,        0,        0       ],
    [0,         0,        a,        0       ],
    [0,         0,        0,        a       ]
])

# ============================================================
# 4. Metric and inverse metric
# ============================================================

g_down = sp.simplify(e.T * eta * e)

g_up = sp.simplify(g_down.inv())

# ============================================================
# 5. Electromagnetic field tensor F_mu_nu
# ============================================================
#
# Convention:
#
# A_mu = (-phi/c, A_i)
# x^0 = ct
#
# Therefore:
#
# F_0i = -E_i/c
# F_12 = B_z
# F_13 = -B_y
# F_23 = B_x

F_down = sp.Matrix([
    [0,      -Ex/c,  -Ey/c,  -Ez/c],
    [Ex/c,   0,      Bz,     -By  ],
    [Ey/c,  -Bz,     0,       Bx  ],
    [Ez/c,   By,    -Bx,      0   ]
])

# ============================================================
# 6. Raise both indices: F^{mu nu}
# ============================================================
#
# F^{mu nu} = g^{mu alpha} g^{nu beta} F_{alpha beta}
#
# Matrix convention:
#
# F_up[mu, nu] = g_up[mu, alpha] F_down[alpha, beta] g_up[nu, beta]
#
# Since g_up is symmetric:
#
# F_up = g_up * F_down * g_up

F_up = sp.simplify(g_up * F_down * g_up)

# ============================================================
# 7. Mixed tensor F^nu_rho
# ============================================================
#
# F^nu_rho = g^{nu lambda} F_{lambda rho}
#
# Matrix convention:
#
# F_mixed[nu, rho] = g_up[nu, lambda] F_down[lambda, rho]

F_mixed = sp.simplify(g_up * F_down)

# ============================================================
# 8. Scalar F_{rho sigma} F^{rho sigma}
# ============================================================

F_scalar = sp.simplify(
    sum(
        F_down[rho, sigma] * F_up[rho, sigma]
        for rho in range(4)
        for sigma in range(4)
    )
)

# ============================================================
# 9. Energy-momentum tensor Theta_EM^{mu nu}
# ============================================================
#
# Theta^{mu nu}
# =
# 1/mu0 [
#     F^{mu rho} F^nu_rho
#     - 1/4 g^{mu nu} F_{rho sigma}F^{rho sigma}
# ]

Theta = sp.Matrix([
    [
        sp.simplify(
            (
                sum(
                    F_up[mu, rho] * F_mixed[nu, rho]
                    for rho in range(4)
                )
                -
                sp.Rational(1, 4) * g_up[mu, nu] * F_scalar
            ) / mu0
        )
        for nu in range(4)
    ]
    for mu in range(4)
])

sp.simplify(Theta)

⎡   ⎛⎛             ⎛   ⎛              2                 2     2        2       ↪
⎢ 2 ⎜⎝(c - κ_EM⋅φ)⋅⎝Eₓ⋅⎝A₁⋅A₂⋅E_y⋅κ_EM  + A₁⋅A₃⋅E_z⋅κ_EM  - A₂ ⋅Eₓ⋅κ_EM  + A₂⋅ ↪
⎢c ⋅⎜───────────────────────────────────────────────────────────────────────── ↪
⎢   ⎝                                                                          ↪
⎢───────────────────────────────────────────────────────────────────────────── ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢                           

In [ ]:
eps = sp.symbols("eps")

subs_linear = {
    kappa: eps*kappa
}

Theta_eps = Theta.subs(subs_linear)

Theta_linear = Theta_eps.applyfunc(
    lambda expr: sp.series(expr, eps, 0, 2).removeO()
)

Theta_linear = Theta_linear.subs(eps, 1)

sp.simplify(Theta_linear)

⎡ 2   ⎛  2      2      2⎞      2        ⎛  2      2      2⎞    3 ⎛  2      2   ↪
⎢a ⋅c⋅⎝Eₓ  + E_y  + E_z ⎠ + 4⋅a ⋅κ_EM⋅φ⋅⎝Eₓ  + E_y  + E_z ⎠ + c ⋅⎝Bₓ  + B_y  + ↪
⎢───────────────────────────────────────────────────────────────────────────── ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢                    2                           2      ⎛     2         2      ↪
⎢                 2⋅a ⋅c⋅(-B_y⋅E_z + B_z⋅E_y) + a ⋅κ_EM⋅⎝A₁⋅Eₓ  + A₁⋅E_y  + A₁ ↪
⎢                 ──────────────────────────────────────────────────────────── ↪
⎢                                                                              ↪
⎢                           